# Inserindo os dados no banco

- Os dados capturados estão presentes na plataforma Kaggle. (https://www.kaggle.com/aayushmishra1512/twitchdata)
- O dataset contém os principais canais da Twitch. Esses dados consistem em coisas diferentes, como número de visualizadores, número de visualizadores ativos, seguidores ganhos e muitas outras colunas relevantes sobre um determinado streamer. Possui 11 colunas diferentes com todas as informações necessárias.



In [ ]:
# Fazendo os imports necessários
import os
import sys
sys.path.append('..')  # Para importar config do diretório pai

import psycopg2 as db
import pandas as pd
from config import DB_CONFIG, SCHEMA, DATA_PATH

In [ ]:
# Capturando o csv
try:
    dataframe = pd.read_csv(f'../{DATA_PATH}')
    print(f"Dados carregados com sucesso. Shape: {dataframe.shape}")
    dataframe.head()
except FileNotFoundError:
    print(f"Erro: Arquivo {DATA_PATH} não encontrado.")
    raise
except Exception as e:
    print(f"Erro ao carregar dados: {e}")
    raise

,Channel,Watch time(Minutes),Stream time(minutes),Peak viewers,Average viewers,Followers,Followers gained,Views gained,Partnered,Mature,Language
0,xQcOW,6196161750,215250,222720,27716,3246298,1734810,93036735,True,False,English
1,summit1g,6091677300,211845,310998,25610,5310163,1370184,89705964,True,False,English
2,Gaules,5644590915,515280,387315,10976,1767635,1023779,102611607,True,True,Portuguese
3,ESL_CSGO,3970318140,517740,300575,7714,3944850,703986,106546942,True,False,English
4,Tfue,3671000070,123660,285644,29602,8938903,2068424,78998587,True,False,English


In [ ]:
# Conectando com o banco
try:
    # Usar configurações do config.py
    conn = db.connect(**DB_CONFIG)
    cur = conn.cursor()
    print("Conexão com o banco estabelecida com sucesso.")
except Exception as e:
    print(f"Erro ao conectar com o banco: {e}")
    raise

In [ ]:
# Query para adicionar no banco de dados
table_name = f"{SCHEMA}.twitchdata"
query = f"""INSERT INTO {table_name} 
(channel, watch_time, stream_time, peak_viewrs, average_viewers, 
followers, followers_gained, views_gained, partnered, mature, lang) 
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"""

# Verificar se a tabela existe e está vazia
cur.execute(f"SELECT COUNT(*) FROM {table_name}")
count = cur.fetchone()[0]
if count > 0:
    print(f"Aviso: A tabela já contém {count} registros. Os dados serão adicionados.")
else:
    print("Tabela vazia. Inserindo dados...")

In [ ]:
# Preparar dados para inserção
data_list = [tuple(row) for row in dataframe.itertuples(index=False)]
print(f"Preparando {len(data_list)} registros para inserção.")

In [ ]:
# Executar inserção em lotes para melhor performance
try:
    batch_size = 1000
    for i in range(0, len(data_list), batch_size):
        batch = data_list[i:i + batch_size]
        cur.executemany(query, batch)
        conn.commit()
        print(f"Inseridos {min(i + batch_size, len(data_list))} de {len(data_list)} registros...")
    
    print("Inserção concluída com sucesso!")
    
except Exception as e:
    print(f"Erro durante a inserção: {e}")
    conn.rollback()
    raise
finally:
    # Fechar conexão
    cur.close()
    conn.close()
    print("Conexão com o banco fechada.")